In [0]:
%run ../control_framework/__init__ 

In [0]:
import time
from pyspark.sql import Row

In [0]:
dbutils.widgets.text("env", "dev", "env")
# dbutils.widgets.text("param_name", "ingestion", "param name")
dbutils.widgets.text("process_name", "aadp_daily_stocks_load", "process name")
dbutils.widgets.text("process_date", "20260716", "process date")
dbutils.widgets.text("run_type", "load", "run type")
dbutils.widgets.text("source", "finnhub_dp", "Source")
dbutils.widgets.text("target", "aadp_finnhub", "Target")
dbutils.widgets.text("execution_type", "consumption_AADP", "execution type")

In [0]:
env = dbutils.widgets.get("env")
# param_name = dbutils.widgets.get("param_name")
process_name = dbutils.widgets.get("process_name")
process_date = dbutils.widgets.get("process_date")
process_date = int(process_date)
run_type = dbutils.widgets.get("run_type")
source = dbutils.widgets.get("source")
target = dbutils.widgets.get("target")
execution_type = dbutils.widgets.get("execution_type")
print(f"env: {env}")
# print(f"param_name: {param_name}")
print(f"process_name: {process_name}")
print(f"process_date: {process_date}")
print(f"run_type: {run_type}")
print(f"source: {source}")
print(f"target: {target}")
print(f"execution_type: {execution_type}")

In [0]:
try:
    comment = register_tgt(target, execution_type)
    insert_log(target, process_name, "target_register","target_register","target_register","target_register",process_date, 'success',comment)
except Exception as e:
    error_msg = str(e).replace("'", "''")
    insert_log(target, process_name, "target_register","target_register","target_register","target_register",process_date, 'failed', error_msg)
    print(f"Error in register_target: {e}")
    dbutils.notebook.exit(f"Error in register_target: {e}")

In [0]:
src_schema1 = f_get_src_schema(source)
src_tbl1 = 's_and_p_500_daily_stock_price'
src_tbl2 = 's_and_p_midcap_400_daily_stock_price'
src_tbl3 = 's_and_p_smallcap_600_daily_stock_price'
# src_schema_master = f_get_src_schema('master')
# src_master_tb1 = 's_and_p_500_dev'
target_schema = f_get_tgt_schema(target)
target_tbl = 'finnhub_daily_stocks'
full_target_table_name = f"{target_schema}.{target_tbl}"
print(src_schema1)
print(src_tbl1)
print(src_tbl2)
print(src_tbl3)
# print(src_schema_master)
# print(src_master_tb1)
print(target_schema)
print(target_tbl)
print(full_target_table_name)

In [0]:
src_tbl_list = [{"source":source,"source_schema":src_schema1,"source_table": src_tbl1},{"source":source,"source_schema":src_schema1,"source_table": src_tbl2},{"source":source,"source_schema":src_schema1,"source_table": src_tbl3}]
tgt_tbl_list = [{"target":target,"target_schema":target_schema,"target_table": target_tbl,'holiday_ind': 'N','weekend_ind': 'N','calender':'','frequency': 'daily'}]

In [0]:
src_tgt_mapping_list = [
    {'source_schema':src_schema1,'source_table':src_tbl1,'source_column_name':'symbol','source_column_datatype':'string','target_schema':target_schema,'target_table':target_tbl,'target_column_name':'stock_symbol','target_column_datatype':'string'},
    {'source_schema':src_schema1,'source_table':src_tbl1,'source_column_name':'process_date','source_column_datatype':'long','target_schema':target_schema,'target_table':target_tbl,'target_column_name':'process_date','target_column_datatype':'long'},
    {'source_schema':src_schema1,'source_table':src_tbl1,'source_column_name':'r_source','source_column_datatype':'string','target_schema':target_schema,'target_table':target_tbl,'target_column_name':'r_source','target_column_datatype':'string'},
    {'source_schema':src_schema1,'source_table':src_tbl1,'source_column_name':'hash_id','source_column_datatype':'string','target_schema':target_schema,'target_table':target_tbl,'target_column_name':'hash_id','target_column_datatype':'string'},
    {'source_schema':src_schema1,'source_table':src_tbl1,'source_column_name':'current_price','source_column_datatype':'double','target_schema':target_schema,'target_table':target_tbl,'target_column_name':'current_price','target_column_datatype':'double'},
    {'source_schema':src_schema1,'source_table':src_tbl1,'source_column_name':'price_change','source_column_datatype':'double','target_schema':target_schema,'target_table':target_tbl,'target_column_name':'price_change','target_column_datatype':'double'},
    {'source_schema':src_schema1,'source_table':src_tbl1,'source_column_name':'day_high_price','source_column_datatype':'double','target_schema':target_schema,'target_table':target_tbl,'target_column_name':'day_high_price','target_column_datatype':'double'},
    {'source_schema':src_schema1,'source_table':src_tbl1,'source_column_name':'day_low_price','source_column_datatype':'double','target_schema':target_schema,'target_table':target_tbl,'target_column_name':'day_low_price','target_column_datatype':'double'},
    {'source_schema':src_schema1,'source_table':src_tbl1,'source_column_name':'opening_price','source_column_datatype':'double','target_schema':target_schema,'target_table':target_tbl,'target_column_name':'opening_price','target_column_datatype':'double'},
    {'source_schema':src_schema1,'source_table':src_tbl1,'source_column_name':'previous_close_price','source_column_datatype':'double','target_schema':target_schema,'target_table':target_tbl,'target_column_name':'previous_close_price','target_column_datatype':'double'},
    {'source_schema':src_schema1,'source_table':src_tbl1,'source_column_name':'quote_timestamp','source_column_datatype':'long','target_schema':target_schema,'target_table':target_tbl,'target_column_name':'quote_timestamp','target_column_datatype':'timestamp'},
    {'source_schema':src_schema1,'source_table':src_tbl1,'source_column_name':'trade_date','source_column_datatype':'long','target_schema':target_schema,'target_table':target_tbl,'target_column_name':'trade_date','target_column_datatype':'date'},
    {'source_schema':src_schema1,'source_table':src_tbl1,'source_column_name':'price_change_percent','source_column_datatype':'double','target_schema':target_schema,'target_table':target_tbl,'target_column_name':'price_change_percent','target_column_datatype':'double'},
    {'source_schema':src_schema1,'source_table':src_tbl1,'source_column_name':'r_target','source_column_datatype':'string','target_schema':target_schema,'target_table':target_tbl,'target_column_name':'r_target','target_column_datatype':'string'},
    {'source_schema':src_schema1,'source_table':src_tbl1,'source_column_name':'opening_price,previous_close_price','source_column_datatype':'double','target_schema':target_schema,'target_table':target_tbl,'target_column_name':'open_gap','target_column_datatype':'double'},
    {'source_schema':src_schema1,'source_table':src_tbl1,'source_column_name':'opening_price,previous_close_price','source_column_datatype':'double','target_schema':target_schema,'target_table':target_tbl,'target_column_name':'open_gap_percent','target_column_datatype':'double'},
    {'source_schema':src_schema1,'source_table':src_tbl1,'source_column_name':'day_high_price,day_low_price','source_column_datatype':'double','target_schema':target_schema,'target_table':target_tbl,'target_column_name':'high_low_gap','target_column_datatype':'double'},
    {'source_schema':src_schema1,'source_table':src_tbl1,'source_column_name':'day_high_price,day_low_price','source_column_datatype':'double','target_schema':target_schema,'target_table':target_tbl,'target_column_name':'high_low_gap_percent','target_column_datatype':'double'},
    {'source_schema':src_schema1,'source_table':src_tbl1,'source_column_name':'day_high_price,current_price','source_column_datatype':'double','target_schema':target_schema,'target_table':target_tbl,'target_column_name':'high_current_gap','target_column_datatype':'double'},
    {'source_schema':src_schema1,'source_table':src_tbl1,'source_column_name':'day_high_price,current_price','source_column_datatype':'double','target_schema':target_schema,'target_table':target_tbl,'target_column_name':'high_current_gap_percent','target_column_datatype':'double'},
    {'source_schema':src_schema1,'source_table':src_tbl1,'source_column_name':'day_low_price,current_price','source_column_datatype':'double','target_schema':target_schema,'target_table':target_tbl,'target_column_name':'low_current_gap','target_column_datatype':'double'},
    {'source_schema':src_schema1,'source_table':src_tbl1,'source_column_name':'day_low_price,current_price','source_column_datatype':'double','target_schema':target_schema,'target_table':target_tbl,'target_column_name':'low_current_gap_percent','target_column_datatype':'double'},
    {'source_schema':src_schema1,'source_table':src_tbl1,'source_column_name':'close_price','source_column_datatype':'double','target_schema':target_schema,'target_table':target_tbl,'target_column_name':'close_price_6_days_ago','target_column_datatype':'double'},
    {'source_schema':src_schema1,'source_table':src_tbl1,'source_column_name':'close_price,close_price_6_days_ago','source_column_datatype':'double','target_schema':target_schema,'target_table':target_tbl,'target_column_name':'weekly_moving','target_column_datatype':'double'},
    {'source_schema':src_schema1,'source_table':src_tbl1,'source_column_name':'close_price,close_price_6_days_ago','source_column_datatype':'double','target_schema':target_schema,'target_table':target_tbl,'target_column_name':'weekly_moving_percentage','target_column_datatype':'double'},
    {'source_schema':src_schema1,'source_table':src_tbl2,'source_column_name':'symbol','source_column_datatype':'string','target_schema':target_schema,'target_table':target_tbl,'target_column_name':'stock_symbol','target_column_datatype':'string'},
    {'source_schema':src_schema1,'source_table':src_tbl2,'source_column_name':'process_date','source_column_datatype':'long','target_schema':target_schema,'target_table':target_tbl,'target_column_name':'process_date','target_column_datatype':'long'},
    {'source_schema':src_schema1,'source_table':src_tbl2,'source_column_name':'r_source','source_column_datatype':'string','target_schema':target_schema,'target_table':target_tbl,'target_column_name':'r_source','target_column_datatype':'string'},
    {'source_schema':src_schema1,'source_table':src_tbl2,'source_column_name':'hash_id','source_column_datatype':'string','target_schema':target_schema,'target_table':target_tbl,'target_column_name':'hash_id','target_column_datatype':'string'},
    {'source_schema':src_schema1,'source_table':src_tbl2,'source_column_name':'current_price','source_column_datatype':'double','target_schema':target_schema,'target_table':target_tbl,'target_column_name':'current_price','target_column_datatype':'double'},
    {'source_schema':src_schema1,'source_table':src_tbl2,'source_column_name':'price_change','source_column_datatype':'double','target_schema':target_schema,'target_table':target_tbl,'target_column_name':'price_change','target_column_datatype':'double'},
    {'source_schema':src_schema1,'source_table':src_tbl2,'source_column_name':'day_high_price','source_column_datatype':'double','target_schema':target_schema,'target_table':target_tbl,'target_column_name':'day_high_price','target_column_datatype':'double'},
    {'source_schema':src_schema1,'source_table':src_tbl2,'source_column_name':'day_low_price','source_column_datatype':'double','target_schema':target_schema,'target_table':target_tbl,'target_column_name':'day_low_price','target_column_datatype':'double'},
    {'source_schema':src_schema1,'source_table':src_tbl2,'source_column_name':'opening_price','source_column_datatype':'double','target_schema':target_schema,'target_table':target_tbl,'target_column_name':'opening_price','target_column_datatype':'double'},
    {'source_schema':src_schema1,'source_table':src_tbl2,'source_column_name':'previous_close_price','source_column_datatype':'double','target_schema':target_schema,'target_table':target_tbl,'target_column_name':'previous_close_price','target_column_datatype':'double'},
    {'source_schema':src_schema1,'source_table':src_tbl2,'source_column_name':'quote_timestamp','source_column_datatype':'long','target_schema':target_schema,'target_table':target_tbl,'target_column_name':'quote_timestamp','target_column_datatype':'timestamp'},
    {'source_schema':src_schema1,'source_table':src_tbl2,'source_column_name':'trade_date','source_column_datatype':'long','target_schema':target_schema,'target_table':target_tbl,'target_column_name':'trade_date','target_column_datatype':'date'},
    {'source_schema':src_schema1,'source_table':src_tbl2,'source_column_name':'price_change_percent','source_column_datatype':'double','target_schema':target_schema,'target_table':target_tbl,'target_column_name':'price_change_percent','target_column_datatype':'double'},
    {'source_schema':src_schema1,'source_table':src_tbl2,'source_column_name':'r_target','source_column_datatype':'string','target_schema':target_schema,'target_table':target_tbl,'target_column_name':'r_target','target_column_datatype':'string'},
    {'source_schema':src_schema1,'source_table':src_tbl2,'source_column_name':'opening_price,previous_close_price','source_column_datatype':'double','target_schema':target_schema,'target_table':target_tbl,'target_column_name':'open_gap','target_column_datatype':'double'},
    {'source_schema':src_schema1,'source_table':src_tbl2,'source_column_name':'opening_price,previous_close_price','source_column_datatype':'double','target_schema':target_schema,'target_table':target_tbl,'target_column_name':'open_gap_percent','target_column_datatype':'double'},
    {'source_schema':src_schema1,'source_table':src_tbl2,'source_column_name':'day_high_price,day_low_price','source_column_datatype':'double','target_schema':target_schema,'target_table':target_tbl,'target_column_name':'high_low_gap','target_column_datatype':'double'},
    {'source_schema':src_schema1,'source_table':src_tbl2,'source_column_name':'day_high_price,day_low_price','source_column_datatype':'double','target_schema':target_schema,'target_table':target_tbl,'target_column_name':'high_low_gap_percent','target_column_datatype':'double'},
    {'source_schema':src_schema1,'source_table':src_tbl2,'source_column_name':'day_high_price,current_price','source_column_datatype':'double','target_schema':target_schema,'target_table':target_tbl,'target_column_name':'high_current_gap','target_column_datatype':'double'},
    {'source_schema':src_schema1,'source_table':src_tbl2,'source_column_name':'day_high_price,current_price','source_column_datatype':'double','target_schema':target_schema,'target_table':target_tbl,'target_column_name':'high_current_gap_percent','target_column_datatype':'double'},
    {'source_schema':src_schema1,'source_table':src_tbl2,'source_column_name':'day_low_price,current_price','source_column_datatype':'double','target_schema':target_schema,'target_table':target_tbl,'target_column_name':'low_current_gap','target_column_datatype':'double'},
    {'source_schema':src_schema1,'source_table':src_tbl2,'source_column_name':'day_low_price,current_price','source_column_datatype':'double','target_schema':target_schema,'target_table':target_tbl,'target_column_name':'low_current_gap_percent','target_column_datatype':'double'},
    {'source_schema':src_schema1,'source_table':src_tbl2,'source_column_name':'close_price','source_column_datatype':'double','target_schema':target_schema,'target_table':target_tbl,'target_column_name':'close_price_6_days_ago','target_column_datatype':'double'},
    {'source_schema':src_schema1,'source_table':src_tbl2,'source_column_name':'close_price,close_price_6_days_ago','source_column_datatype':'double','target_schema':target_schema,'target_table':target_tbl,'target_column_name':'weekly_moving','target_column_datatype':'double'},
    {'source_schema':src_schema1,'source_table':src_tbl2,'source_column_name':'close_price,close_price_6_days_ago','source_column_datatype':'double','target_schema':target_schema,'target_table':target_tbl,'target_column_name':'weekly_moving_percentage','target_column_datatype':'double'},
    {'source_schema':src_schema1,'source_table':src_tbl3,'source_column_name':'symbol','source_column_datatype':'string','target_schema':target_schema,'target_table':target_tbl,'target_column_name':'stock_symbol','target_column_datatype':'string'},
    {'source_schema':src_schema1,'source_table':src_tbl3,'source_column_name':'process_date','source_column_datatype':'long','target_schema':target_schema,'target_table':target_tbl,'target_column_name':'process_date','target_column_datatype':'long'},
    {'source_schema':src_schema1,'source_table':src_tbl3,'source_column_name':'r_source','source_column_datatype':'string','target_schema':target_schema,'target_table':target_tbl,'target_column_name':'r_source','target_column_datatype':'string'},
    {'source_schema':src_schema1,'source_table':src_tbl3,'source_column_name':'hash_id','source_column_datatype':'string','target_schema':target_schema,'target_table':target_tbl,'target_column_name':'hash_id','target_column_datatype':'string'},
    {'source_schema':src_schema1,'source_table':src_tbl3,'source_column_name':'current_price','source_column_datatype':'double','target_schema':target_schema,'target_table':target_tbl,'target_column_name':'current_price','target_column_datatype':'double'},
    {'source_schema':src_schema1,'source_table':src_tbl3,'source_column_name':'price_change','source_column_datatype':'double','target_schema':target_schema,'target_table':target_tbl,'target_column_name':'price_change','target_column_datatype':'double'},
    {'source_schema':src_schema1,'source_table':src_tbl3,'source_column_name':'day_high_price','source_column_datatype':'double','target_schema':target_schema,'target_table':target_tbl,'target_column_name':'day_high_price','target_column_datatype':'double'},
    {'source_schema':src_schema1,'source_table':src_tbl3,'source_column_name':'day_low_price','source_column_datatype':'double','target_schema':target_schema,'target_table':target_tbl,'target_column_name':'day_low_price','target_column_datatype':'double'},
    {'source_schema':src_schema1,'source_table':src_tbl3,'source_column_name':'opening_price','source_column_datatype':'double','target_schema':target_schema,'target_table':target_tbl,'target_column_name':'opening_price','target_column_datatype':'double'},
    {'source_schema':src_schema1,'source_table':src_tbl3,'source_column_name':'previous_close_price','source_column_datatype':'double','target_schema':target_schema,'target_table':target_tbl,'target_column_name':'previous_close_price','target_column_datatype':'double'},
    {'source_schema':src_schema1,'source_table':src_tbl3,'source_column_name':'quote_timestamp','source_column_datatype':'long','target_schema':target_schema,'target_table':target_tbl,'target_column_name':'quote_timestamp','target_column_datatype':'timestamp'},
    {'source_schema':src_schema1,'source_table':src_tbl3,'source_column_name':'trade_date','source_column_datatype':'long','target_schema':target_schema,'target_table':target_tbl,'target_column_name':'trade_date','target_column_datatype':'date'},
    {'source_schema':src_schema1,'source_table':src_tbl3,'source_column_name':'price_change_percent','source_column_datatype':'double','target_schema':target_schema,'target_table':target_tbl,'target_column_name':'price_change_percent','target_column_datatype':'double'},
    {'source_schema':src_schema1,'source_table':src_tbl3,'source_column_name':'r_target','source_column_datatype':'string','target_schema':target_schema,'target_table':target_tbl,'target_column_name':'r_target','target_column_datatype':'string'},
    {'source_schema':src_schema1,'source_table':src_tbl3,'source_column_name':'opening_price,previous_close_price','source_column_datatype':'double','target_schema':target_schema,'target_table':target_tbl,'target_column_name':'open_gap','target_column_datatype':'double'},
    {'source_schema':src_schema1,'source_table':src_tbl3,'source_column_name':'opening_price,previous_close_price','source_column_datatype':'double','target_schema':target_schema,'target_table':target_tbl,'target_column_name':'open_gap_percent','target_column_datatype':'double'},
    {'source_schema':src_schema1,'source_table':src_tbl3,'source_column_name':'day_high_price,day_low_price','source_column_datatype':'double','target_schema':target_schema,'target_table':target_tbl,'target_column_name':'high_low_gap','target_column_datatype':'double'},
    {'source_schema':src_schema1,'source_table':src_tbl3,'source_column_name':'day_high_price,day_low_price','source_column_datatype':'double','target_schema':target_schema,'target_table':target_tbl,'target_column_name':'high_low_gap_percent','target_column_datatype':'double'},
    {'source_schema':src_schema1,'source_table':src_tbl3,'source_column_name':'day_high_price,current_price','source_column_datatype':'double','target_schema':target_schema,'target_table':target_tbl,'target_column_name':'high_current_gap','target_column_datatype':'double'},
    {'source_schema':src_schema1,'source_table':src_tbl3,'source_column_name':'day_high_price,current_price','source_column_datatype':'double','target_schema':target_schema,'target_table':target_tbl,'target_column_name':'high_current_gap_percent','target_column_datatype':'double'},
    {'source_schema':src_schema1,'source_table':src_tbl3,'source_column_name':'day_low_price,current_price','source_column_datatype':'double','target_schema':target_schema,'target_table':target_tbl,'target_column_name':'low_current_gap','target_column_datatype':'double'},
    {'source_schema':src_schema1,'source_table':src_tbl3,'source_column_name':'day_low_price,current_price','source_column_datatype':'double','target_schema':target_schema,'target_table':target_tbl,'target_column_name':'low_current_gap_percent','target_column_datatype':'double'},
    {'source_schema':src_schema1,'source_table':src_tbl3,'source_column_name':'close_price','source_column_datatype':'double','target_schema':target_schema,'target_table':target_tbl,'target_column_name':'close_price_6_days_ago','target_column_datatype':'double'},
    {'source_schema':src_schema1,'source_table':src_tbl3,'source_column_name':'close_price,close_price_6_days_ago','source_column_datatype':'double','target_schema':target_schema,'target_table':target_tbl,'target_column_name':'weekly_moving','target_column_datatype':'double'},
    {'source_schema':src_schema1,'source_table':src_tbl3,'source_column_name':'close_price,close_price_6_days_ago','source_column_datatype':'double','target_schema':target_schema,'target_table':target_tbl,'target_column_name':'weekly_moving_percentage','target_column_datatype':'double'}
]

In [0]:
if run_type == 'load_metadata':
    status = run_load_metadata(src_tbl_list,tgt_tbl_list,target,process_name,src_tgt_mapping_list,process_date)
    dbutils.notebook.exit(status)


In [0]:
from pyspark.sql.types import StringType, DoubleType, LongType, StructType, StructField, TimestampType,DateType

schema_structure = {
    "stock_symbol": StringType(),
    "process_date": LongType(),
    "r_source": StringType(),
    "hash_id": StringType(),
    "current_price": DoubleType(),
    "price_change": DoubleType(),
    "day_high_price": DoubleType(),
    "day_low_price": DoubleType(),
    "opening_price": DoubleType(),
    "previous_close_price": DoubleType(),
    "quote_timestamp": TimestampType(),
    "trade_date": DateType(),
    "price_change_percent": DoubleType(),
    "r_target": StringType(),
    "open_gap": DoubleType(),
    "open_gap_percent": DoubleType(),
    "high_low_gap": DoubleType(),
    "high_low_gap_percent": DoubleType(),
    "high_current_gap": DoubleType(),
    "high_current_gap_percent": DoubleType(),
    "low_current_gap": DoubleType(),
    "low_current_gap_percent": DoubleType(),
    "close_price_6_days_ago": DoubleType(),
    "weekly_moving": DoubleType(),
    "weekly_moving_percentage": DoubleType(),
}

In [0]:
delta_logic = f"""process_date = {process_date}"""

In [0]:
if run_type == 'full':
    create_tgt_tbl_log = create_target_table(full_target_table_name,schema_structure,is_history='Y')
    if  create_tgt_tbl_log == f"success":
        process_status = 'success'
    else:
        process_status = 'failed'
    log = str(create_tgt_tbl_log).replace("'", "''")
    print(process_status)
    insert_log (target, process_name, 'create_tgt_tbl','create_tgt_tbl', target_schema, target_tbl, process_date,process_status, log)

    delta_logic = f"""process_date <= {process_date}"""

In [0]:
column_fetch = f"""
stock_symbol,
process_date,
'SADP' as r_source,
hash_id,
current_price,
price_change,
day_high_price,
day_low_price,
opening_price,
previous_close_price,
quote_timestamp,
trade_date,
price_change_percent,
'aadp_finnhub' as r_target,
CASE WHEN (previous_close_price - opening_price) IS NULL OR (previous_close_price - opening_price) = 0 THEN NULL ELSE (previous_close_price - opening_price) END AS open_gap,
CASE WHEN (previous_close_price - opening_price) IS NULL OR (previous_close_price - opening_price) = 0 THEN NULL ELSE (((previous_close_price - opening_price)/opening_price)*100) END AS open_gap_percent,
CASE WHEN (day_high_price - day_low_price) IS NULL OR (day_high_price - day_low_price) = 0 THEN NULL ELSE (day_high_price - day_low_price) END AS high_low_gap,
CASE WHEN (day_high_price - day_low_price) IS NULL OR (day_high_price - day_low_price) = 0 THEN NULL ELSE (((day_high_price - day_low_price)/day_low_price)*100) END AS high_low_gap_percent,
CASE WHEN (day_high_price - current_price) IS NULL OR (day_high_price - current_price) = 0 THEN NULL ELSE (day_high_price - current_price) END AS high_current_gap,
CASE WHEN (day_high_price - current_price) IS NULL OR (day_high_price - current_price) = 0 THEN NULL ELSE (((day_high_price - current_price)/current_price)*100) END AS high_current_gap_percent,
CASE WHEN (current_price - day_low_price) IS NULL OR (current_price - day_low_price) = 0 THEN NULL ELSE (current_price - day_low_price) END AS low_current_gap,
CASE WHEN (current_price - day_low_price) IS NULL OR (current_price - day_low_price) = 0 THEN NULL ELSE (((current_price - day_low_price)/current_price)*100) END AS low_current_gap_percent,
LAG(current_price, 6) OVER (PARTITION BY stock_symbol ORDER BY trade_date) AS close_price_6_days_ago,
CASE
    WHEN LAG(current_price, 6) OVER (PARTITION BY stock_symbol ORDER BY trade_date) IS NOT NULL
    THEN current_price - LAG(current_price, 6) OVER (PARTITION BY stock_symbol ORDER BY trade_date)
    ELSE NULL
END AS weekly_moving,
CASE
    WHEN LAG(current_price, 6) OVER (PARTITION BY stock_symbol ORDER BY trade_date) IS NOT NULL
    THEN ((current_price - LAG(current_price, 6) OVER (PARTITION BY stock_symbol ORDER BY trade_date))/LAG(current_price, 6) OVER (PARTITION BY stock_symbol ORDER BY trade_date))*100
    ELSE NULL
END AS weekly_moving_percentage
"""

In [0]:
v_sel_main1 = f"""with s as (
  select *, row_number() over (partition by hash_id order by process_date desc) as rn
  from {src_schema1}.{src_tbl1} 
  union 
  select *, row_number() over (partition by hash_id order by process_date desc) as rn
  from
  {src_schema1}.{src_tbl2}
  union 
  select *, row_number() over (partition by hash_id order by process_date desc) as rn
  from
  {src_schema1}.{src_tbl3}
  where {delta_logic}
),
src as (
  select {column_fetch} from s where rn = 1
),
tgt as (
  select * from {target_schema}.{target_tbl}_history
  where r_target = '{target}'
),
main as (
  select src.*
  from src
  left anti join tgt
  on src.hash_id = tgt.hash_id
)
select *  from main
"""
print(v_sel_main1)

In [0]:
insert_table_query_history = f"""insert into {target_schema}.{target_tbl}_history
(stock_symbol,
  process_date,
  r_source,
  hash_id,
  current_price,
  price_change,
  day_high_price,
  day_low_price,
  opening_price,
  previous_close_price,
  quote_timestamp,
  trade_date,
  price_change_percent,
  r_target,
  open_gap,
  open_gap_percent,
  high_low_gap,
  high_low_gap_percent,
  high_current_gap,
  high_current_gap_percent,
  low_current_gap,
  low_current_gap_percent,
  close_price_6_days_ago,
  weekly_moving,
  weekly_moving_percentage)
 {v_sel_main1}"""
print(insert_table_query_history)

In [0]:
truncat_tble_query = f"""truncate table {target_schema}.{target_tbl}"""
print(truncat_tble_query)


In [0]:
insert_table_query = f"""insert into {target_schema}.{target_tbl}
select * from {target_schema}.{target_tbl}_history
where r_target = '{target}'
and process_date = {process_date}"""
print(insert_table_query)

In [0]:
if run_type == "rollback":
    try:
        comment = rollback_run(tgt_tbl_list,process_date)
        insert_log(target, process_name, 'rollback', 'rollback', target_schema, target_tbl, process_date, 'success', str(comment).replace("'", "''"))
    except Exception as e:
        error_msg = str(e).replace("'", "''")
        insert_log(target, process_name, 'rollback', 'rollback', target_schema, target_tbl, process_date, 'failed', error_msg)
        comment = f"Error in rollback: {e}"
    dbutils.notebook.exit(f"{comment}")

In [0]:
if run_type == 'full' or run_type == 'load':
    try:
        spark.sql(f"""{insert_table_query_history}""")
        spark.sql(f"""{truncat_tble_query}""")
        spark.sql(f"""{insert_table_query}""")
        insert_log(target, process_name, 'data_consumption', 'data_consumption', target_schema, target_tbl, process_date, 'success', 'Data consumption completed')
        e = "success"
    except Exception as e:
        error_msg = str(e).replace("'", "''")
        insert_log(target, process_name, 'data_consumption', 'data_consumption', target_schema, target_tbl, process_date, 'failed', error_msg)
        e = f"Error in data_consumption: {e}"
    dbutils.notebook.exit(f"{e}")